# SIGMark: ProPainter and DiffuEraser

Repository: [https://github.com/JeremyZhao1998/SIGMark-release](https://github.com/JeremyZhao1998/SIGMark-release)

Paper: [https://arxiv.org/abs/2603.02882](https://arxiv.org/abs/2603.02882)

Inspected commit: `3713243f2e002cb21bbc3da896094d39b2b528c9`

Watermark stage: `in_generation`
Domain: video

This method is coupled to generation, so the notebook uses only author-provided or author-generated native data. It does not fake a MOSE embedding.

The two removers receive the exact same watermarked frames and binary mask. One official DiffuEraser execution is used: its internal ProPainter prior is preserved as the ProPainter result, and its final refinement is the DiffuEraser result. Inputs shorter than 22 frames are padded by repeating only the last frame/mask to 24 for inference, then both reported outputs are trimmed to the authentic length.

The repository contains the authors' VBench2 prompt text but no generated watermarked MP4. This notebook generates a small native HunyuanVideo-T2V sample with SIGMark, preserving the maintained state and ground-truth message. The paper/repository state that the unquantized path needs roughly 50 GB GPU memory. A deterministic center-region mask is recorded because no object mask is released.


## Authenticity contract

- Watermark preparation: Generate a SIGMark video with the official HunyuanVideo or HunyuanVideo-I2V integration and create a matching object mask for the generated frames.
- Checkpoint status: REQUIRES HunyuanVideo model files, at least 48 GB CPU RAM, and approximately 50 GB GPU RAM for the authors' unquantized single-GPU path. Prompt text is included in the repository.
- Protocol note: SIGMark is an in-generation watermark.  An ordinary or post-hoc-watermarked MOSE clip is not a valid substitute.
- No result cell fabricates metrics. Native extraction must finish and write its JSON before values are displayed.
- A method without a released spatial detector reports bit recovery only; it is not assigned a proxy localizer.


In [ ]:
from pathlib import Path
import json
import os
import shlex
import subprocess
import sys
from IPython.display import JSON, Video, display

def find_repo_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "scripts/remaining_watermark_methods.py").is_file():
            return candidate
    raise FileNotFoundError("Start Jupyter inside the paper-reproductions-task1 checkout")

ROOT = find_repo_root()
SCRIPTS = ROOT / "scripts"

def run(command, *, cwd=None, env=None):
    command = [str(value) for value in command]
    print("+", shlex.join(command))
    subprocess.run(command, cwd=cwd, env=env, check=True)

def require(path, description):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {description}: {path}")
    return path

def require_commit(repository, expected):
    actual = subprocess.check_output(["git", "-C", str(repository), "rev-parse", "HEAD"], text=True).strip()
    if actual != expected:
        raise RuntimeError(f"Expected {expected} in {repository}, found {actual}")

def show_results(run_root):
    run_root = Path(run_root)
    for name in ("watermarked.mp4", "aligned-mask.mp4"):
        path = run_root / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    attack = run_root / "attack"
    for name in ("input-watermarked.mp4", "gt-mask.mp4", "propainter.mp4", "diffueraser.mp4"):
        path = attack / name
        if path.is_file():
            print(name)
            display(Video(str(path), embed=True, html_attributes="controls muted loop"))
    for path in sorted(run_root.rglob("summary.json")):
        print(path.relative_to(run_root))
        display(JSON(json.loads(path.read_text())))

print("Repository:", ROOT)


In [ ]:
SIGMARK = Path("/path/to/SIGMark-release")
SIGMARK_PYTHON = SIGMARK / ".venv/bin/python"
TORCHRUN = SIGMARK / ".venv/bin/torchrun"
MODEL_BASE = Path("/path/to/pretrained_models")
DIFFUERASER = Path("/path/to/DiffuEraser")
DIFFUERASER_PYTHON = DIFFUERASER / ".venv/bin/python"
GPU = "0"
RUN_ROOT = ROOT / "runs/remaining-watermarks/sigmark/native-vbench2"
NATIVE = RUN_ROOT / "native-generation"
RUN_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
require_commit(SIGMARK, "3713243f2e002cb21bbc3da896094d39b2b528c9")
require(TORCHRUN, "SIGMark torchrun")
require(MODEL_BASE / "HunyuanVideo-community", "HunyuanVideo-community model")


In [ ]:
if not list(NATIVE.rglob("*.mp4")):
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        TORCHRUN, "--nproc_per_node=1", SIGMARK / "main.py", "--mode=gen",
        "--model_base_path", MODEL_BASE, "--model_name", "HunyuanVideo-community",
        "--prompt_set", "VBench2_aug", "--watermark_method", "sigmark",
        "--ch_factor", "2", "--hw_factor", "8", "--fr_factor", "1",
        "--batch_size", "1", "--num_prompts_per_dimension", "1",
        "--num_videos_per_prompt", "1", "--num_prompts_diversity", "1",
        "--num_videos_per_prompt_diversity", "1", "--small_scale_test", "1",
        "--output_path", NATIVE,
    ], cwd=SIGMARK, env=env)
WATERMARKED_VIDEO = sorted(NATIVE.rglob("*.mp4"))[0]
SAMPLE_KEY = str(WATERMARKED_VIDEO.relative_to(NATIVE).with_suffix(""))
ALIGNED_MASK = RUN_ROOT / "aligned-mask.mp4"
print("Native sample:", WATERMARKED_VIDEO, "key:", SAMPLE_KEY)


In [ ]:
run([
    DIFFUERASER_PYTHON, SCRIPTS / "make_native_removal_mask.py", "center-box",
    "--video", WATERMARKED_VIDEO, "--output-video", ALIGNED_MASK,
    "--output-frames", RUN_ROOT / "aligned-mask-frames",
])


## Run the identical two-method attack

The preparation cell freezes hashes, frame count, dimensions, FPS, mask foreground counts, provenance, and any short-video padding. The next cell runs both methods from that frozen input.


In [ ]:
ATTACK_RUN = RUN_ROOT / "attack"
if not (ATTACK_RUN / "sample.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "prepare_watermark_removal_sample.py", "prepare",
        "--source-video", WATERMARKED_VIDEO, "--source-mask", ALIGNED_MASK,
        "--output", ATTACK_RUN, "--watermark-method", "SIGMark",
        "--sample-id", "first VBench2_aug small-scale sample", "--sample-source", "SIGMark authors' VBench2 prompt set and native HunyuanVideo pipeline",
        "--source-kind", "authors_generated",
    ])
else:
    print("Prepared sample already exists:", ATTACK_RUN / "sample.json")


In [ ]:
if not (ATTACK_RUN / "removal-pair.json").is_file():
    run([
        DIFFUERASER_PYTHON, SCRIPTS / "run_watermark_removal_pair.py",
        "--run", ATTACK_RUN, "--diffueraser", DIFFUERASER,
        "--diffueraser-python", DIFFUERASER_PYTHON,
        "--device", GPU, "--seed", "2025",
    ])
else:
    print("Both removal outputs already exist:", ATTACK_RUN / "removal-pair.json")


In [ ]:
DETECT_CASES = {
    "original": ATTACK_RUN / "input-watermarked.mp4",
    "propainter": ATTACK_RUN / "propainter.mp4",
    "diffueraser": ATTACK_RUN / "diffueraser.mp4",
}
for name, video in DETECT_CASES.items():
    staged = RUN_ROOT / "detection" / name
    if not (staged / "staged-case.json").is_file():
        run([
            DIFFUERASER_PYTHON, SCRIPTS / "stage_sigmark_detection_case.py",
            "--generated-output", NATIVE, "--sample-key", SAMPLE_KEY,
            "--replacement-video", video, "--destination", staged,
        ])
    env = os.environ.copy(); env["CUDA_VISIBLE_DEVICES"] = GPU
    run([
        TORCHRUN, "--nproc_per_node=1", SIGMARK / "main.py", "--mode=extract",
        "--model_base_path", MODEL_BASE, "--model_name", "HunyuanVideo-community",
        "--prompt_set", "VBench2_aug", "--watermark_method", "sigmark",
        "--ch_factor", "2", "--hw_factor", "8", "--fr_factor", "1",
        "--batch_size", "1", "--num_prompts_per_dimension", "1",
        "--num_videos_per_prompt", "1", "--num_prompts_diversity", "1",
        "--num_videos_per_prompt_diversity", "1", "--small_scale_test", "1",
        "--output_path", staged,
    ], cwd=SIGMARK, env=env)


In [ ]:
show_results(RUN_ROOT)
